# Datathon Passos Mágicos — Modelo Preditivo de Risco de Defasagem

Objetivo: prever a **probabilidade de risco** de defasagem (ou piora de defasagem) para apoiar intervenção antecipada.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))
from plots_passos import plot_feature_importance, plot_calibration_curve

df = pd.read_csv(PROJECT_ROOT / "data_processed" / "pede_consolidado.csv")
for c in ["inde","ida","ieg","iaa","ips","ipp","ipv","ian","defasagem","defasagem_next",
          "piora_defasagem_prox_ano","risco_defasagem_atual","risco_defasagem_severo"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print("Shape:", df.shape)
display(df.head())

## 1) Definir target de modelagem

In [ ]:
# Escolha do alvo:
# A) risco atual (mais estável e mais linhas)
# B) piora no próximo ano (mais útil, mas menos linhas)

TARGET = "piora_defasagem_prox_ano" if "piora_defasagem_prox_ano" in df.columns and df["piora_defasagem_prox_ano"].notna().sum() > 50 else "risco_defasagem_atual"
print("Target selecionado:", TARGET)

y = pd.to_numeric(df[TARGET], errors="coerce")
valid_target_mask = y.notna()
model_df = df.loc[valid_target_mask].copy()
y = y.loc[valid_target_mask].astype(int)

print("Linhas para modelagem:", len(model_df))
print("Taxa positiva:", y.mean())

## 2) Selecionar features (sem vazamento)

In [ ]:
# Excluímos colunas-alvo e possíveis vazamentos diretos
leak_cols = {
    TARGET, "risco_defasagem_atual", "risco_defasagem_severo",
    "defasagem_next", "piora_defasagem_prox_ano"
}

candidate_cols = [
    c for c in model_df.columns
    if c not in leak_cols
]

# Preferência por indicadores e contexto escolar
preferred_num = [c for c in ["ida","ieg","iaa","ips","ipp","ipv","ian","inde","mat","por","ing","idade","ano_referencia","defasagem"] if c in candidate_cols]
preferred_cat = [c for c in ["fase","genero","instituicao_de_ensino","turma","pedra","pedra_2023","fase_ideal"] if c in candidate_cols]

# Mantém somente colunas com sinal razoável
use_cols = []
for c in preferred_num + preferred_cat:
    if c in model_df.columns and model_df[c].notna().mean() >= 0.1:
        use_cols.append(c)

X = model_df[use_cols].copy()

numeric_features = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Features numéricas:", numeric_features)
print("Features categóricas:", categorical_features)
display(X.head())

## 3) Split temporal (preferencial) ou holdout aleatório

In [ ]:
if "ano_referencia" in X.columns and X["ano_referencia"].nunique() >= 2:
    anos = sorted(X["ano_referencia"].dropna().unique().tolist())
    train_years = anos[:-1]
    test_year = anos[-1]

    train_mask = X["ano_referencia"].isin(train_years)
    test_mask = X["ano_referencia"].eq(test_year)

    if train_mask.sum() >= 50 and test_mask.sum() >= 20:
        X_train, X_test = X.loc[train_mask], X.loc[test_mask]
        y_train, y_test = y.loc[train_mask], y.loc[test_mask]
        print(f"Split temporal: treino={train_years}, teste={test_year}")
    else:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
        print("Fallback para split aleatório estratificado.")
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    print("Split aleatório estratificado.")

print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())

## 4) Pipeline de pré-processamento + modelos

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features),
    ],
    remainder="drop"
)

models = {
    "log_reg": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "rf": RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1,
    ),
}

results = []
fitted = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ("prep", preprocessor),
        ("clf", clf)
    ])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, p) if len(np.unique(y_test)) > 1 else np.nan
    ap = average_precision_score(y_test, p) if len(np.unique(y_test)) > 1 else np.nan

    results.append({"modelo": name, "roc_auc": auc, "avg_precision": ap})
    fitted[name] = (pipe, p)

results_df = pd.DataFrame(results).sort_values(["roc_auc","avg_precision"], ascending=False)
display(results_df)
best_name = results_df.iloc[0]["modelo"]
best_pipe, best_prob = fitted[best_name]
print("Melhor modelo:", best_name)

## 5) Avaliação e gráficos

In [ ]:
# Curvas ROC e PR
if len(np.unique(y_test)) > 1:
    fpr, tpr, _ = roc_curve(y_test, best_prob)
    prec, rec, _ = precision_recall_curve(y_test, best_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr)
    ax.plot([0,1],[0,1], linestyle="--")
    ax.set_title(f"ROC - {best_name}")
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.grid(alpha=0.2)
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(rec, prec)
    ax.set_title(f"Precision-Recall - {best_name}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.grid(alpha=0.2)
    plt.show()

    plot_calibration_curve(y_test, best_prob, outpath="outputs/figs/ml_calibracao.png")
else:
    print("Teste com uma única classe; métricas limitadas.")

## 6) Threshold operacional e lista de alunos em risco

In [ ]:
threshold = 0.50
pred = (best_prob >= threshold).astype(int)

print("Confusion matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification report:")
print(classification_report(y_test, pred, digits=3))

risco_df = X_test.copy()
risco_df["y_real"] = y_test.values
risco_df["prob_risco"] = best_prob
risco_df["pred_risco"] = pred

# Traz identificador se disponível
if "ra" in model_df.columns:
    risco_df["ra"] = model_df.loc[X_test.index, "ra"].values
if "nome_anonimizado" in model_df.columns:
    risco_df["nome_anonimizado"] = model_df.loc[X_test.index, "nome_anonimizado"].values

display(risco_df.sort_values("prob_risco", ascending=False).head(20))

risco_df.sort_values("prob_risco", ascending=False).to_csv(PROJECT_ROOT / "outputs" / "predicoes_risco_teste.csv", index=False)
print("Arquivo salvo: outputs/predicoes_risco_teste.csv")

## 7) Importância das variáveis

In [ ]:
# Para RandomForest, conseguimos importância direta.
if best_name == "rf":
    prep = best_pipe.named_steps["prep"]
    clf = best_pipe.named_steps["clf"]

    # nomes das features após one-hot
    feature_names = []
    if numeric_features:
        feature_names.extend(numeric_features)
    if categorical_features:
        ohe = prep.named_transformers_["cat"].named_steps["onehot"]
        cat_names = ohe.get_feature_names_out(categorical_features).tolist()
        feature_names.extend(cat_names)

    importances = pd.Series(clf.feature_importances_, index=feature_names).sort_values(ascending=False)
    display(importances.head(25))
    plot_feature_importance(importances, outpath="outputs/figs/ml_feature_importance.png")
else:
    # fallback: coeficientes da regressão logística
    prep = best_pipe.named_steps["prep"]
    clf = best_pipe.named_steps["clf"]

    feature_names = []
    if numeric_features:
        feature_names.extend(numeric_features)
    if categorical_features:
        ohe = prep.named_transformers_["cat"].named_steps["onehot"]
        feature_names.extend(ohe.get_feature_names_out(categorical_features).tolist())

    coefs = pd.Series(np.abs(clf.coef_[0]), index=feature_names).sort_values(ascending=False)
    display(coefs.head(25))
    plot_feature_importance(coefs, outpath="outputs/figs/ml_feature_importance.png")

## 8) Como levar para a apresentação (slide executivo)

- **Probabilidade de risco** por aluno (lista priorizada)
- **Top fatores associados** ao risco (ex.: IEG, IPS, IDA, IAA)
- **Threshold operacional** (ex.: 0.40 para aumentar recall)
- **Ações sugeridas por faixa de risco** (alto, médio, baixo)